In [1]:
# Importing the required libraries
from pathlib import Path
import gc
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
# Setting the project folders
current_folder = Path.cwd()
project_folder = current_folder.parent if current_folder.name == 'notebooks' else current_folder

data_path = project_folder / 'data' / 'processed' / 'churn_model_dataset.csv'
models_folder = project_folder / 'models'
probability_path = project_folder / 'data' / 'processed' / 'customer_churn_probabilities.csv'

models_folder.mkdir(parents=True, exist_ok=True)

print('Dataset:', data_path)
print('Models folder:', models_folder)

Dataset: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\processed\churn_model_dataset.csv
Models folder: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\models


In [ ]:
# Loading the model dataset
df = pd.read_csv(data_path)

print('Rows:', len(df))
print('Columns:', len(df.columns))
df.head()

Rows: 35193
Columns: 36


,customer_id,country,age_group,acquisition_channel,primary_device,tenure_months,plan_type,auto_renew_enabled,discount_applied,billing_cycle,orders_last_30_days,orders_last_90_days,spend_last_90_days,average_order_value,days_since_last_order,late_delivery_rate,return_rate,shipping_fee_saved,watch_minutes_last_30_days,watch_minutes_last_90_days,video_sessions_last_90_days,days_since_last_video_activity,average_completion_rate,payment_failures_last_90_days,payment_retries_last_90_days,last_payment_failed,support_tickets_last_90_days,average_resolution_hours,average_satisfaction_score,repeat_contact_rate,benefits_used_count,total_active_days,days_since_last_activity,engagement_score,customer_segment,churn_next_30_days
0,CUST025026,India,45-54,Paid Advertising,Mobile,8.2,Prime Standard,No,Promo Code,Annual,0,1,15.66,15.66,56,0.0000,0.0,3.90,0.0,0.0,0,149,0.0000,0,0,0,0,NaN,NaN,0.0,3,1,56,19.00,Low-Engagement Members,1
1,CUST020499,United States,55-64,Organic Search,Mobile,11.3,Prime Standard,Yes,No Discount,Annual,0,1,81.49,81.49,47,0.0000,0.0,6.19,0.0,0.0,0,141,0.0000,0,0,0,0,NaN,NaN,0.0,2,1,47,14.00,Low-Engagement Members,0
2,CUST001658,India,55-64,Paid Advertising,Game Console,0.8,Prime Standard,Yes,Retention Offer,Annual,3,3,94.70,31.57,16,0.3333,0.0,11.54,175.0,175.0,4,1,0.4865,0,0,0,1,6.75,3.0,0.0,4,5,1,40.92,Service-Risk Members,0
3,CUST044227,Japan,18-24,Social Media,Fire TV,21.9,Prime Standard,Yes,No Discount,Monthly,0,1,23.84,23.84,64,0.0000,0.0,3.85,207.5,344.0,7,15,0.6440,0,0,0,0,NaN,NaN,0.0,3,5,15,35.73,Regular Members,0
4,CUST001102,India,35-44,Organic Search,Mobile,11.3,Prime Standard,No,No Discount,Annual,1,3,104.43,34.81,3,0.0000,0.0,11.80,608.6,1267.7,16,14,0.7636,0,0,0,0,NaN,NaN,0.0,3,10,3,70.00,Multi-Benefit Power Users,0


In [4]:
# Checking the target balance
target_summary = df['churn_next_30_days'].value_counts().sort_index().to_frame('customers')
target_summary['percentage'] = (
    target_summary['customers'] / len(df) * 100
).round(2)

target_summary

,customers,percentage
churn_next_30_days,,
0,34210,97.21
1,983,2.79


In [5]:
# Separating the customer ID, features and target
customer_ids = df['customer_id'].copy()
y = df['churn_next_30_days'].copy()

leakage_columns = [
    'customer_id',
    'membership_status',
    'cancellation_date',
    'cancellation_reason',
    'membership_end_date',
    'churn_flag',
]

columns_to_remove = [
    column for column in leakage_columns
    if column in df.columns
]

X = df.drop(
    columns=columns_to_remove + ['churn_next_30_days']
)

print('Features:', X.shape[1])
print('Target:', y.name)
print('Removed columns:', columns_to_remove)

Features: 34
Target: churn_next_30_days
Removed columns: ['customer_id']


In [6]:
# Spliting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Training churn rate:', round(y_train.mean() * 100, 2), '%')
print('Testing churn rate:', round(y_test.mean() * 100, 2), '%')

Training rows: 28154
Testing rows: 7039
Training churn rate: 2.79 %
Testing churn rate: 2.8 %


In [7]:
# Separatin numeric and categorical columns
numeric_columns = X.select_dtypes(include='number').columns.tolist()
categorical_columns = X.select_dtypes(exclude='number').columns.tolist()

print('Numeric columns:', len(numeric_columns))
print('Categorical columns:', len(categorical_columns))
print()
print('Categorical columns:')
print(categorical_columns)

Numeric columns: 25
Categorical columns: 9

Categorical columns:
['country', 'age_group', 'acquisition_channel', 'primary_device', 'plan_type', 'auto_renew_enabled', 'discount_applied', 'billing_cycle', 'customer_segment']


In [8]:
# Preparing numeric and categorical features
numeric_steps = Pipeline(
    steps=[
        ('fill_missing', SimpleImputer(strategy='median')),
        ('standardise', StandardScaler()),
    ]
)

categorical_steps = Pipeline(
    steps=[
        ('fill_missing', SimpleImputer(strategy='most_frequent')),
        (
            'one_hot_encode',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_steps, numeric_columns),
        ('categorical', categorical_steps, categorical_columns),
    ]
)

In [9]:
# Creating three simple classification models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=80,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

print('Models:', list(models.keys()))

Models: ['Logistic Regression', 'Random Forest', 'Gradient Boosting']


In [10]:
# Training and comparing the models
model_results = []
confusion_matrices = {}

for model_name, model in models.items():
    model_pipeline = Pipeline(
        steps=[
            ('preprocessor', clone(preprocessor)),
            ('model', clone(model)),
        ]
    )

    if model_name == 'Gradient Boosting':
        sample_weights = compute_sample_weight(
            class_weight='balanced',
            y=y_train,
        )
        model_pipeline.fit(
            X_train,
            y_train,
            model__sample_weight=sample_weights,
        )
    else:
        model_pipeline.fit(X_train, y_train)

    predictions = model_pipeline.predict(X_test)
    probabilities = model_pipeline.predict_proba(X_test)[:, 1]

    model_results.append(
        {
            'model': model_name,
            'accuracy': accuracy_score(y_test, predictions),
            'precision': precision_score(
                y_test, predictions, zero_division=0
            ),
            'recall': recall_score(y_test, predictions),
            'f1_score': f1_score(y_test, predictions),
            'roc_auc': roc_auc_score(y_test, probabilities),
        }
    )

    confusion_matrices[model_name] = confusion_matrix(
        y_test, predictions
    )

    del model_pipeline
    gc.collect()

metrics = pd.DataFrame(model_results)

for column in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
    metrics[column] = metrics[column].round(4)

metrics

,model,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.8240,0.1197,0.8325,0.2093,0.8991
1,Random Forest,0.9246,0.2007,0.5685,0.2967,0.9070
2,Gradient Boosting,0.8317,0.1275,0.8579,0.2219,0.9135


In [11]:
# Saving the model comparison metrics
metrics_path = models_folder / 'model_metrics.csv'
metrics.to_csv(metrics_path, index=False)

print('Saved:', metrics_path)

Saved: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\models\model_metrics.csv


In [12]:
# Showing the confusion matrices
for model_name, matrix in confusion_matrices.items():
    matrix_table = pd.DataFrame(
        matrix,
        index=['Actual: Did Not Churn', 'Actual: Churned'],
        columns=['Predicted: Did Not Churn', 'Predicted: Churned'],
    )
    print(model_name)
    display(matrix_table)

Logistic Regression


,Predicted: Did Not Churn,Predicted: Churned
Actual: Did Not Churn,5636,1206
Actual: Churned,33,164


Random Forest


,Predicted: Did Not Churn,Predicted: Churned
Actual: Did Not Churn,6396,446
Actual: Churned,85,112


Gradient Boosting


,Predicted: Did Not Churn,Predicted: Churned
Actual: Did Not Churn,5685,1157
Actual: Churned,28,169


In [13]:
# Selecting the model using recall, F1, ROC-AUC and precision
metrics['selection_score'] = (
    metrics['recall'] * 0.40
    + metrics['f1_score'] * 0.25
    + metrics['roc_auc'] * 0.25
    + metrics['precision'] * 0.10
).round(4)

best_model_name = metrics.sort_values(
    'selection_score', ascending=False
).iloc[0]['model']

print('Selected model:', best_model_name)
metrics.sort_values('selection_score', ascending=False)

Selected model: Gradient Boosting


,model,accuracy,precision,recall,f1_score,roc_auc,selection_score
2,Gradient Boosting,0.8317,0.1275,0.8579,0.2219,0.9135,0.6398
0,Logistic Regression,0.8240,0.1197,0.8325,0.2093,0.8991,0.6221
1,Random Forest,0.9246,0.2007,0.5685,0.2967,0.9070,0.5484


In [14]:
# Training the selected model again on the training data
best_model = Pipeline(
    steps=[
        ('preprocessor', clone(preprocessor)),
        ('model', clone(models[best_model_name])),
    ]
)

if best_model_name == 'Gradient Boosting':
    sample_weights = compute_sample_weight(
        class_weight='balanced',
        y=y_train,
    )
    best_model.fit(
        X_train,
        y_train,
        model__sample_weight=sample_weights,
    )
else:
    best_model.fit(X_train, y_train)

best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]

selected_metrics = pd.DataFrame(
    {
        'metric': [
            'Accuracy',
            'Precision',
            'Recall',
            'F1 Score',
            'ROC-AUC',
        ],
        'value': [
            accuracy_score(y_test, best_predictions),
            precision_score(y_test, best_predictions, zero_division=0),
            recall_score(y_test, best_predictions),
            f1_score(y_test, best_predictions),
            roc_auc_score(y_test, best_probabilities),
        ],
    }
)

selected_metrics['value'] = selected_metrics['value'].round(4)
selected_metrics

,metric,value
0,Accuracy,0.8317
1,Precision,0.1275
2,Recall,0.8579
3,F1 Score,0.2219
4,ROC-AUC,0.9135


In [15]:
# Retraining the selected model using all available rows
final_model = Pipeline(
    steps=[
        ('preprocessor', clone(preprocessor)),
        ('model', clone(models[best_model_name])),
    ]
)

if best_model_name == 'Gradient Boosting':
    all_sample_weights = compute_sample_weight(
        class_weight='balanced',
        y=y,
    )
    final_model.fit(
        X,
        y,
        model__sample_weight=all_sample_weights,
    )
else:
    final_model.fit(X, y)

model_path = models_folder / 'churn_model.joblib'
joblib.dump(final_model, model_path)

print('Saved:', model_path)

Saved: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\models\churn_model.joblib


In [16]:
# Creating one churn probability for every eligible customer
all_probabilities = final_model.predict_proba(X)[:, 1]
all_predictions = (all_probabilities >= 0.50).astype(int)

customer_probabilities = pd.DataFrame(
    {
        'customer_id': customer_ids,
        'churn_probability': np.round(all_probabilities, 4),
        'predicted_churn': all_predictions,
        'actual_churn_next_30_days': y,
    }
)

customer_probabilities.to_csv(probability_path, index=False)

print('Saved:', probability_path)
customer_probabilities.sort_values(
    'churn_probability', ascending=False
).head(10)

Saved: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\processed\customer_churn_probabilities.csv


,customer_id,churn_probability,predicted_churn,actual_churn_next_30_days
2176,CUST017606,0.9366,1,0
11290,CUST045698,0.9351,1,0
7583,CUST037854,0.9347,1,1
32162,CUST024901,0.9341,1,1
34600,CUST041040,0.9341,1,1
28566,CUST036234,0.9330,1,1
14575,CUST029307,0.9319,1,1
7555,CUST015542,0.9319,1,1
30543,CUST015186,0.9316,1,1
32630,CUST010089,0.9299,1,0


In [17]:
# Final check
print('Selected model:', best_model_name)
print('Model file exists:', model_path.exists())
print('Metrics file exists:', metrics_path.exists())
print('Probability file exists:', probability_path.exists())
print('Probability rows:', len(customer_probabilities))
print('Unique customers:', customer_probabilities['customer_id'].nunique())
print('Minimum probability:', customer_probabilities['churn_probability'].min())
print('Maximum probability:', customer_probabilities['churn_probability'].max())

Selected model: Gradient Boosting
Model file exists: True
Metrics file exists: True
Probability file exists: True
Probability rows: 35193
Unique customers: 35193
Minimum probability: 0.0309
Maximum probability: 0.9366
